# ASSN — CNN + Random Forest
Trening sieci CNN od podstaw na obrazach szkieletów ASL. Augmentacja danych treningowych, podział train/val/test i klasyfikacja Random Forest.

In [ ]:
import sys
from pathlib import Path

DATASET = Path("dataset")
COUNT = 0
AUGMENTATIONS = 2
EPOCHS = 20
BATCH_SIZE = 32
IMAGE_SIZE = 224
SKIP_COREML = False


## Augmentacja obrazu

In [ ]:
import matplotlib.pyplot as plt
from tools.build_coreml import ASLDataset, collect_samples

paths, labels = collect_samples(DATASET, COUNT)
examples = ASLDataset(paths[:1], labels[:1], augmentations=3, image_size=IMAGE_SIZE)
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
for i, ax in enumerate(axes):
    tensor, label = examples[i]
    ax.imshow(tensor.permute(1, 2, 0))
    ax.set_title(f"{label} | {'Oryginał' if i == 0 else 'Augmentacja'}")
    ax.axis("off")
plt.show()


## Trening i ewaluacja

In [ ]:
import subprocess

command = [
    sys.executable, "tools/build_coreml.py",
    "--train", str(DATASET),
    "--count", str(COUNT),
    "--image-size", str(IMAGE_SIZE),
    "--augmentations", str(AUGMENTATIONS),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
]
if SKIP_COREML:
    command.append("--skip-coreml")
subprocess.run(command, check=True)


## Wyniki

In [ ]:
import json
import pandas as pd

folder = Path("tools/trained")
results = json.loads((folder / "results.json").read_text(encoding="utf-8"))
pd.DataFrame([{k: v for k, v in results.items() if k.endswith("_test") }])


In [ ]:
history = pd.read_json(folder / "history.json")
history[["train_loss", "val_loss"]].plot(title="CNN — loss")
history[["train_accuracy", "val_accuracy"]].plot(title="CNN — accuracy")
